In [118]:
# Importando as bibliotecas necessárias
import pandas as pd
from pathlib import Path
import numpy as np

In [119]:
# Importando a base de dados
df = pd.read_csv('data/Superstore.csv',encoding='latin1')
df_vendas = df.copy()

# ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode','Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State','Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category','Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

# Nomes das colunas em português
# ['ID da Linha', 'ID do Pedido', 'Data do Pedido', 'Data de Envio', 'Modo de Envio','ID do Cliente', 'Nome do Cliente', 'Segmento', 'País', 'Cidade', 'Estado','CEP', 'Região', 'ID do Produto', 'Categoria', 'Subcategoria','Nome do Produto', 'Vendas', 'Quantidade', 'Desconto', 'Lucro']


In [120]:
# Removendo colunas que não serão úteis
df_vendas = df_vendas.drop(columns=['Row ID'])

# alterando o tipo da coluna
df_vendas['Ship Date'] = pd.to_datetime(df_vendas['Ship Date'],format='%d-%m-%Y')
df_vendas['Order Date'] = pd.to_datetime(df_vendas['Order Date'], format='%d-%m-%Y')


In [121]:
display(df_vendas.info())
display(df_vendas.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Order ID       9994 non-null   object        
 1   Order Date     9994 non-null   datetime64[ns]
 2   Ship Date      9994 non-null   datetime64[ns]
 3   Ship Mode      9994 non-null   object        
 4   Customer ID    9994 non-null   object        
 5   Customer Name  9994 non-null   object        
 6   Segment        9994 non-null   object        
 7   Country        9994 non-null   object        
 8   City           9994 non-null   object        
 9   State          9994 non-null   object        
 10  Postal Code    9994 non-null   int64         
 11  Region         9994 non-null   object        
 12  Product ID     9994 non-null   object        
 13  Category       9994 non-null   object        
 14  Sub-Category   9994 non-null   object        
 15  Product Name   9994 n

None

,Order Date,Ship Date,Postal Code,Sales,Quantity,Discount,Profit
count,9994,9994,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,2013-04-30 19:20:02.401441024,2013-05-04 18:20:49.229537792,55190.379428,229.858001,3.789574,0.156203,28.656896
min,2011-01-04 00:00:00,2011-01-08 00:00:00,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,2012-05-23 00:00:00,2012-05-27 00:00:00,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,2013-06-27 00:00:00,2013-06-30 00:00:00,56430.500000,54.490000,3.000000,0.200000,8.666500
75%,2014-05-15 00:00:00,2014-05-19 00:00:00,90008.000000,209.940000,5.000000,0.200000,29.364000
max,2014-12-31 00:00:00,2015-01-06 00:00:00,99301.000000,22638.480000,14.000000,0.800000,8399.976000
std,NaN,NaN,32063.693350,623.245101,2.225110,0.206452,234.260108


In [122]:
# Análise exploratória

# Dimensões iniciais do df_vendas
display(df_vendas.shape)

# Valores nulos
nulos = df_vendas.isnull().sum()
nulos_pct = (nulos / len(df_vendas) * 100).round(2)
df_nulos = pd.DataFrame({'Qtd':nulos, 'Percentual':nulos_pct} ) 

display(df_nulos)

# duplicatas exatas
duplicatas_exatas = df_vendas.duplicated().sum()
display(f'Duplicada exata: {duplicatas_exatas}')

# Removendo a duplicata exata
df_vendas = df_vendas.drop_duplicates()

# Checar duplicatas por Order ID + Product ID (não só linha inteira)
mascara = df_vendas.duplicated(subset=['Order ID', 'Product ID'], keep=False)
duplicatas = mascara.sum()
display(f'Linhas duplicadas por Order ID + Product ID: {duplicatas}')

# Calculando o número total de grupos 'Order ID' + 'Product ID'
suspeitos = df_vendas[mascara]
pares_unicos = suspeitos.groupby(['Order ID', 'Product ID']).ngroups
print(f'{len(suspeitos)} linhas envolvendo {pares_unicos} combinações únicas de Order ID + Product ID')

# Verificação de valores inconsistentes - Vendas ou quantidades negativas não fazem sentido
print(df_vendas[df_vendas['Sales'] <= 0] ) 
print(df_vendas[df_vendas['Quantity'] <= 0] )
print(df_vendas[df_vendas['Discount'] < 0] )


(9994, 20)

,Qtd,Percentual
Order ID,0,0.0
Order Date,0,0.0
Ship Date,0,0.0
Ship Mode,0,0.0
Customer ID,0,0.0
Customer Name,0,0.0
Segment,0,0.0
Country,0,0.0
City,0,0.0
State,0,0.0


'Duplicada exata: 1'

'Linhas duplicadas por Order ID + Product ID: 14'

14 linhas envolvendo 7 combinações únicas de Order ID + Product ID
Empty DataFrame
Columns: [Order ID, Order Date, Ship Date, Ship Mode, Customer ID, Customer Name, Segment, Country, City, State, Postal Code, Region, Product ID, Category, Sub-Category, Product Name, Sales, Quantity, Discount, Profit]
Index: []
Empty DataFrame
Columns: [Order ID, Order Date, Ship Date, Ship Mode, Customer ID, Customer Name, Segment, Country, City, State, Postal Code, Region, Product ID, Category, Sub-Category, Product Name, Sales, Quantity, Discount, Profit]
Index: []
Empty DataFrame
Columns: [Order ID, Order Date, Ship Date, Ship Mode, Customer ID, Customer Name, Segment, Country, City, State, Postal Code, Region, Product ID, Category, Sub-Category, Product Name, Sales, Quantity, Discount, Profit]
Index: []


In [123]:
# Criando colunas

"""
- Os valores da coluna Sales já estão com o desconto (Discount) aplicado.
- A coluna total_cost já inclui o desconto (Sales) embutido, então ela reflete o custo por unidade vendida na transação, não necessariamente o custo de aquisição do produto em si, que seria fixo independente do desconto.

"""

# Preço unitário
df_vendas['unit_price'] = df_vendas.apply(
    lambda r: r['Sales'] / r['Quantity'] if r['Discount'] == 1 
    else r['Sales'] / (r['Quantity'] * (1 - r['Discount'])), axis=1
)
# Venada sem desconto
df_vendas['sale_without_discount'] = df_vendas['unit_price'] * df_vendas['Quantity']

# Custo total
df_vendas['total_cost'] = df_vendas['Sales'] - df_vendas['Profit']
# Custo unitário
df_vendas['unit_cost'] = df_vendas['total_cost'] / df_vendas['Quantity']

In [124]:
# Verificando a base após a criação das colunas

display(df_vendas.describe())

# Verificando a existência de valores nulos
display(df_vendas.isnull().sum())

# Verificando a existência de valores infinitos
display(np.isinf(df_vendas.loc[:, 'Sales':]).sum())

print(df_vendas[df_vendas['Sales'] <= 0] ) 
print(df_vendas[df_vendas['Quantity'] <= 0] )
print(df_vendas[df_vendas['Discount'] < 0] )
print(df_vendas[df_vendas['unit_price'] <= 0] ) 
print(df_vendas[df_vendas['sale_without_discount'] <= 0] )
print(df_vendas[df_vendas['total_cost'] <= 0] ) 
print(df_vendas[df_vendas['unit_cost'] <= 0] )

,Order Date,Ship Date,Postal Code,Sales,Quantity,Discount,Profit,unit_price,sale_without_discount,total_cost,unit_cost
count,9993,9993,9993.000000,9993.000000,9993.000000,9993.000000,9993.000000,9993.000000,9993.000000,9993.000000,9993.000000
mean,2013-04-30 21:06:30.153107200,2013-05-04 20:07:16.625637888,55191.576403,229.852846,3.789753,0.156188,28.660971,75.558577,286.553896,201.191875,53.110831
min,2011-01-04 00:00:00,2011-01-08 00:00:00,1040.000000,0.444000,1.000000,0.000000,-6599.978000,0.990000,0.990000,0.554400,0.544500
25%,2012-05-23 00:00:00,2012-05-27 00:00:00,23223.000000,17.280000,2.000000,0.000000,1.731000,6.480000,21.360000,12.688200,3.473600
50%,2013-06-27 00:00:00,2013-06-30 00:00:00,56560.000000,54.480000,3.000000,0.200000,8.671000,19.980000,64.960000,41.664000,12.933600
75%,2014-05-15 00:00:00,2014-05-19 00:00:00,90008.000000,209.940000,5.000000,0.200000,29.364000,76.980000,251.910000,182.220000,54.522000
max,2014-12-31 00:00:00,2015-01-06 00:00:00,99301.000000,22638.480000,14.000000,0.800000,8399.976000,7546.160000,45276.960000,24449.558400,4074.926400
std,NaN,NaN,32065.074478,623.276074,2.225149,0.206457,234.271476,188.966751,864.138239,550.866205,122.253930


Order ID                 0
Order Date               0
Ship Date                0
Ship Mode                0
Customer ID              0
Customer Name            0
Segment                  0
Country                  0
City                     0
State                    0
Postal Code              0
Region                   0
Product ID               0
Category                 0
Sub-Category             0
Product Name             0
Sales                    0
Quantity                 0
Discount                 0
Profit                   0
unit_price               0
sale_without_discount    0
total_cost               0
unit_cost                0
dtype: int64

Sales                    0
Quantity                 0
Discount                 0
Profit                   0
unit_price               0
sale_without_discount    0
total_cost               0
unit_cost                0
dtype: int64

Empty DataFrame
Columns: [Order ID, Order Date, Ship Date, Ship Mode, Customer ID, Customer Name, Segment, Country, City, State, Postal Code, Region, Product ID, Category, Sub-Category, Product Name, Sales, Quantity, Discount, Profit, unit_price, sale_without_discount, total_cost, unit_cost]
Index: []

[0 rows x 24 columns]
Empty DataFrame
Columns: [Order ID, Order Date, Ship Date, Ship Mode, Customer ID, Customer Name, Segment, Country, City, State, Postal Code, Region, Product ID, Category, Sub-Category, Product Name, Sales, Quantity, Discount, Profit, unit_price, sale_without_discount, total_cost, unit_cost]
Index: []

[0 rows x 24 columns]
Empty DataFrame
Columns: [Order ID, Order Date, Ship Date, Ship Mode, Customer ID, Customer Name, Segment, Country, City, State, Postal Code, Region, Product ID, Category, Sub-Category, Product Name, Sales, Quantity, Discount, Profit, unit_price, sale_without_discount, total_cost, unit_cost]
Index: []

[0 rows x 24 columns]
Empty DataFrame
Column

In [125]:
# Exportar a base de dados
caminho = Path.cwd() / 'data'
nome_arquivo = 'Superstore_tratada.csv'
caminho_completo = caminho / nome_arquivo

if Path.exists(caminho_completo):
  df_vendas.to_csv(caminho_completo, sep=',', index=False)
else:
  df_vendas.to_csv(caminho_completo, sep=',', index=False)

## Conclusões da Análise exploratória (Base de dados original)
- A base de dados possui 9994 linhas e 24 colunas.
- A 'Row ID' é uma coluna de índice. Não tem relevancia para a análise (deletada).
- A base de dados não apresenta nenhum valor nulo
- Possui 1 duplicata exata (deletada)
- Possui 15 duplicatas por 'Order ID' + 'Product ID'
  - Após remover a duplicata exata, restaram 14 linhas envolvendo 
    7 combinações únicas de Order ID + Product ID, consideradas 
    itens separados legítimos pois Quantity, Sales e Profit 
    apresentam valores diferentes e proporcionais entre si.
- As colunas 'Sales', 'Quantity' e 'Discount' não apresentam valores 
  inconsistentes — nenhum registro com Sales ou Quantity <= 0, 
  e nenhum Discount negativo.

## Base de dados Tratada
- Após a crição das colunas 'unit_price', 'sale_without_discount',	'total_cost' e	'unit_cost' a base não apresentou nenhum valor inconsistente (menor ou igual a 0)